# Random Disclosure Parity Game: Dynamic Programming Solution

## Game Definition

> *The Random Disclosure Parity Game is a two-player zero-sum game with four moves:*
>
> 1. **Player I** selects an integer from $\{1, 2\}$.
> 2. **Referee** tosses a fair coin. Heads $\to$ Player II is informed of Player I's choice; Tails $\to$ Player II receives no information.
> 3. **Player II** selects an integer from $\{3, 4\}$.
> 4. **Referee** draws an integer from $\{1, 2, 3\}$ with probabilities $\{0.4, 0.2, 0.4\}$.
>
> The sum of moves 1, 3, and 4 is computed. If even, Player II pays Player I \$1; if odd, Player I pays Player II \$1.

In this notebook, we solve the game **exactly** using **Dynamic Programming** (DP). Unlike sampling-based methods (Monte Carlo, Q-learning, gradient bandits), DP computes optimal values and policies by solving the Bellman equations directly — no simulation needed.

### Challenges for DP in This Game

The Addition game has perfect information — every state is fully observed, and backward induction on the game tree suffices. The Parity game is harder:

1. **Imperfect information**: When the coin is tails, Player II does not know Player I's choice. Player II's strategy at information set $T$ must be the same regardless of the hidden choice.
2. **Chance nodes**: The coin flip and referee draw inject randomness into the game tree.
3. **Mixed-strategy equilibrium**: The optimal strategy requires randomisation — DP must solve a matrix game subproblem, not just pick a maximum.

Our approach: **stage-by-stage backward induction**, solving subgames at each information set. Where imperfect information creates a simultaneous-move subgame, we solve the resulting matrix game via linear programming.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, Tuple, List
import random

plt.style.use('dark_background')
plt.rcParams['font.family'] = 'monospace'
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.facecolor'] = '#1e1e2e'
plt.rcParams['figure.facecolor'] = '#11111b'
plt.rcParams['axes.edgecolor'] = '#6c7086'
plt.rcParams['axes.labelcolor'] = '#89b4fa'
plt.rcParams['xtick.color'] = '#a6adc8'
plt.rcParams['ytick.color'] = '#a6adc8'
plt.rcParams['text.color'] = '#cdd6f4'
plt.rcParams['grid.color'] = '#313244'
plt.rcParams['grid.alpha'] = 0.5

## 1. Bellman Equations for the Parity Game

We work **backward** through four stages of the game tree.

### Stage 4 — Referee Draw (Chance Node)

Given Player I chose $i$ and Player II chose $j$, the referee draws $r \in \{1,2,3\}$ with probabilities $\{0.4, 0.2, 0.4\}$. The terminal payoff to Player I is:

$$\text{payoff}(i,j,r) = \begin{cases} +1 & \text{if } i+j+r \text{ is even} \\ -1 & \text{if } i+j+r \text{ is odd} \end{cases}$$

The **conditional expected payoff** (integrating over the referee) is:

$$g(i,j) = \sum_{r \in \{1,2,3\}} P(r)\,\text{payoff}(i,j,r)$$

### Stage 3 — Player II's Decision (Information Sets)

Player II has three information sets:

- **$H_1$** (heads, PI chose 1): PII knows $i=1$, picks $j$ to **minimise** $g(1,j)$
- **$H_2$** (heads, PI chose 2): PII knows $i=2$, picks $j$ to **minimise** $g(2,j)$
- **$T$** (tails): PII does **not** know $i$. This creates a **simultaneous-move subgame** — PII must choose $j$ (or mix over $j$) without knowing PI's hidden choice. The optimal solution is the minimax of the $2 \times 2$ matrix $[g(i,j)]$.

### Stage 2 — Coin Flip (Chance Node)

$$V_\text{coin}(i) = \frac{1}{2}\,V_{H}(i) + \frac{1}{2}\,V_T$$

where $V_H(i)$ is the value under heads (PII knows $i$) and $V_T$ is the value of the tails subgame.

### Stage 1 — Player I's Decision

$$V^* = \max_i V_\text{coin}(i) = \max_i \left[\frac{1}{2}\,V_H(i) + \frac{1}{2}\,V_T\right]$$

If $V_\text{coin}(1) = V_\text{coin}(2)$, Player I is indifferent and may mix.

## 2. Game Parameters

In [ ]:
PI_ACTIONS = [1, 2]
PII_ACTIONS = [3, 4]
REFEREE_OUTCOMES = [(1, 0.4), (2, 0.2), (3, 0.4)]
COIN_PROB = 0.5

print("Random Disclosure Parity Game")
print("═" * 50)
print(f"  PI actions:   {PI_ACTIONS}")
print(f"  PII actions:  {PII_ACTIONS}")
print(f"  Referee:      {{r: P(r) for r, P(r) in {REFEREE_OUTCOMES}}}")
print(f"  Coin P(H):    {COIN_PROB}")
print(f"  Payoff:       +1 if sum even, −1 if sum odd")

## 3. Stage 4 — Terminal Payoffs and Conditional Expected Values

We first compute every terminal payoff $\text{payoff}(i,j,r)$ and then the conditional expectations $g(i,j)$.

In [ ]:
def terminal_payoff(i: int, j: int, r: int) -> float:
    """Payoff to Player I for a specific (i, j, r) triple."""
    return 1.0 if (i + j + r) % 2 == 0 else -1.0


def conditional_payoff(i: int, j: int) -> float:
    """Expected payoff to PI, integrating over the referee draw."""
    return sum(p * terminal_payoff(i, j, r)
               for r, p in REFEREE_OUTCOMES)


print("STAGE 4: Terminal Payoff Table  payoff(i, j, r)")
print("═" * 60)
print(f"{'(i,j)':>8}", end="")
for r, _ in REFEREE_OUTCOMES:
    print(f"{'r='+str(r):>10}", end="")
print(f"{'g(i,j)':>12}")
print("─" * 60)

g_matrix = {}
for i in PI_ACTIONS:
    for j in PII_ACTIONS:
        g_val = conditional_payoff(i, j)
        g_matrix[(i, j)] = g_val
        row = f"({i},{j})"
        print(f"{row:>8}", end="")
        for r, _ in REFEREE_OUTCOMES:
            print(f"{terminal_payoff(i,j,r):>+10.0f}", end="")
        print(f"{g_val:>+12.1f}")

print("\n  g(i,j) = Σ_r P(r) · payoff(i,j,r)")
print("  Parity pattern: same-parity (i+j) even → most r give +1")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
fig.suptitle('Stage 4: Terminal Payoffs and Conditional Expectations',
             fontsize=14, fontweight='bold', color='#89b4fa')

# Full terminal payoff table
combos = [(i, j, r) for i in PI_ACTIONS for j in PII_ACTIONS
          for r, _ in REFEREE_OUTCOMES]
data = np.array([[terminal_payoff(i, j, r) for r, _ in REFEREE_OUTCOMES]
                  for i in PI_ACTIONS for j in PII_ACTIONS])
labels_y = [f'i={i}, j={j}' for i in PI_ACTIONS for j in PII_ACTIONS]
labels_x = [f'r={r}\nP={p}' for r, p in REFEREE_OUTCOMES]

im1 = ax1.imshow(data, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto')
ax1.set_yticks(range(len(labels_y)))
ax1.set_yticklabels(labels_y)
ax1.set_xticks(range(len(labels_x)))
ax1.set_xticklabels(labels_x)
ax1.set_title('payoff(i, j, r)', color='#89b4fa')
for ii in range(data.shape[0]):
    for jj in range(data.shape[1]):
        c = 'white' if abs(data[ii, jj]) > 0.5 else 'black'
        ax1.text(jj, ii, f'{data[ii,jj]:+.0f}', ha='center',
                 va='center', color=c, fontweight='bold')
plt.colorbar(im1, ax=ax1, shrink=0.8)

# g(i,j) matrix
g_arr = np.array([[conditional_payoff(i, j) for j in PII_ACTIONS]
                   for i in PI_ACTIONS])
im2 = ax2.imshow(g_arr, cmap='RdYlGn', vmin=-1, vmax=1, aspect='equal')
ax2.set_yticks(range(len(PI_ACTIONS)))
ax2.set_yticklabels([f'i={i}' for i in PI_ACTIONS])
ax2.set_xticks(range(len(PII_ACTIONS)))
ax2.set_xticklabels([f'j={j}' for j in PII_ACTIONS])
ax2.set_title('g(i, j) = E_r[payoff]', color='#89b4fa')
for ii in range(g_arr.shape[0]):
    for jj in range(g_arr.shape[1]):
        c = 'white' if abs(g_arr[ii, jj]) > 0.5 else 'black'
        ax2.text(jj, ii, f'{g_arr[ii,jj]:+.1f}', ha='center',
                 va='center', color=c, fontsize=14, fontweight='bold')
plt.colorbar(im2, ax=ax2, shrink=0.8)

plt.tight_layout()
plt.show()

## 4. Stage 3 — Player II's Optimal Response at Each Information Set

### Perfect-information sets ($H_1$, $H_2$)

Player II observes $i$ and solves a simple minimisation:

$$V_H(i) = \min_{j \in \{3,4\}} g(i, j)$$

### Imperfect-information set ($T$) — Matrix Game

Player II does not know $i$. Player I chose $i$ at Stage 1, so from the perspective of the $T$-subgame this is a simultaneous-move $2 \times 2$ zero-sum game with payoff matrix $[g(i,j)]$.

We solve this via the **minimax theorem** (von Neumann, 1928). For a $2 \times 2$ matrix, the mixed-strategy Nash equilibrium can be computed analytically.

In [ ]:
def solve_heads_subgame(i: int) -> dict:
    """Solve PII's decision at info set H_i (perfect information)."""
    payoffs = {j: conditional_payoff(i, j) for j in PII_ACTIONS}
    best_j = min(PII_ACTIONS, key=lambda j: payoffs[j])
    return {
        'info_set': f'H{i}',
        'payoffs': payoffs,
        'optimal_j': best_j,
        'value': payoffs[best_j],
        'pii_mix': {j: (1.0 if j == best_j else 0.0)
                    for j in PII_ACTIONS}
    }


def solve_2x2_matrix_game(M: np.ndarray) -> dict:
    """Solve a 2×2 zero-sum game analytically.

    M[i][j] = payoff to the row player (PI).
    Returns mixed strategy NE and game value.
    """
    a, b = M[0, 0], M[0, 1]
    c, d = M[1, 0], M[1, 1]

    # Check for saddle point (pure strategy equilibrium)
    row_mins = [min(M[i, :]) for i in range(2)]
    col_maxs = [max(M[:, j]) for j in range(2)]
    maximin = max(row_mins)
    minimax = min(col_maxs)

    if maximin == minimax:
        i_star = row_mins.index(maximin)
        j_star = col_maxs.index(minimax)
        p = [0.0, 0.0]; p[i_star] = 1.0
        q = [0.0, 0.0]; q[j_star] = 1.0
        return {'value': maximin,
                'pi_mix': dict(zip(PI_ACTIONS, p)),
                'pii_mix': dict(zip(PII_ACTIONS, q)),
                'type': 'pure'}

    # Mixed strategy: p1 = (d - c) / (a - b - c + d)
    denom = a - b - c + d
    p1 = (d - c) / denom
    q1 = (d - b) / denom
    value = (a * d - b * c) / denom
    if value == 0:
        value = 0.0

    return {'value': value,
            'pi_mix': {PI_ACTIONS[0]: p1,
                       PI_ACTIONS[1]: 1 - p1},
            'pii_mix': {PII_ACTIONS[0]: q1,
                        PII_ACTIONS[1]: 1 - q1},
            'type': 'mixed'}


# Solve each subgame
h1_result = solve_heads_subgame(1)
h2_result = solve_heads_subgame(2)

M_tails = np.array([[conditional_payoff(i, j) for j in PII_ACTIONS]
                     for i in PI_ACTIONS])
tails_result = solve_2x2_matrix_game(M_tails)

print("STAGE 3: Player II's Optimal Response")
print("═" * 60)

for res in [h1_result, h2_result]:
    info = res['info_set']
    print(f"\n  Info set {info} (PII knows PI's choice):")
    for j in PII_ACTIONS:
        marker = " ◀ min" if j == res['optimal_j'] else ""
        print(f"    g({info[-1]}, {j}) = {res['payoffs'][j]:+.1f}{marker}")
    print(f"    V_H({info[-1]}) = {res['value']:+.1f}")
    print(f"    PII plays: {res['optimal_j']} (pure)")

print(f"\n  Info set T (PII does not know PI's choice):")
print(f"    Subgame matrix [g(i,j)]:")
print(f"           j=3      j=4")
for idx, i in enumerate(PI_ACTIONS):
    print(f"    i={i}  {M_tails[idx,0]:+6.1f}   {M_tails[idx,1]:+6.1f}")
print(f"    Solution type: {tails_result['type']}")
print(f"    PI mix:  P(1)={tails_result['pi_mix'][1]:.2f}, "
      f"P(2)={tails_result['pi_mix'][2]:.2f}")
print(f"    PII mix: P(3)={tails_result['pii_mix'][3]:.2f}, "
      f"P(4)={tails_result['pii_mix'][4]:.2f}")
print(f"    V_T = {tails_result['value']:+.2f}")

## 5. Stage 2 — Coin Flip (Chance Node)

Integrate over the coin:

$$V_\text{coin}(i) = P(H)\,V_H(i) + P(T)\,V_T$$

In [ ]:
V_T = tails_result['value']

V_coin = {}
for i in PI_ACTIONS:
    V_H_i = solve_heads_subgame(i)['value']
    V_coin[i] = COIN_PROB * V_H_i + (1 - COIN_PROB) * V_T

print("STAGE 2: Coin Flip Integration")
print("═" * 60)
for i in PI_ACTIONS:
    V_H_i = solve_heads_subgame(i)['value']
    print(f"  V_coin({i}) = {COIN_PROB}·V_H({i}) + {1-COIN_PROB}·V_T")
    print(f"           = {COIN_PROB}·({V_H_i:+.1f}) + "
          f"{1-COIN_PROB}·({V_T:+.1f})")
    print(f"           = {V_coin[i]:+.2f}")

## 6. Stage 1 — Player I's Optimal Decision

$$V^* = \max_i V_\text{coin}(i)$$

If $V_\text{coin}(1) = V_\text{coin}(2)$, Player I is indifferent and any mixture is optimal.

In [ ]:
V_star = max(V_coin.values())
pi_indifferent = abs(V_coin[1] - V_coin[2]) < 1e-10

print("STAGE 1: Player I's Decision")
print("═" * 60)
print(f"  V_coin(1) = {V_coin[1]:+.2f}")
print(f"  V_coin(2) = {V_coin[2]:+.2f}")
print(f"  V* = max{{{V_coin[1]:+.2f}, {V_coin[2]:+.2f}}} = {V_star:+.2f}")

if pi_indifferent:
    print(f"\n  PI is INDIFFERENT: V_coin(1) = V_coin(2).")
    print(f"  Any mixture P(1) ∈ [0,1] is optimal for PI.")
    print(f"  Equilibrium convention: P(1) = P(2) = 0.50.")
else:
    best_i = max(PI_ACTIONS, key=lambda i: V_coin[i])
    print(f"\n  PI plays: {best_i} (pure)")

## 7. Complete DP Solution Summary

In [ ]:
print("╔" + "═" * 58 + "╗")
print("║  DYNAMIC PROGRAMMING — COMPLETE SOLUTION" + " " * 17 + "║")
print("╠" + "═" * 58 + "╣")
print(f"║  Game value:     V* = {V_star:+.2f}" + " " * 30 + "║")
print(f"║  (Player II has a ${abs(V_star):.2f} edge per game)" + " " * 17 + "║")
print("╠" + "═" * 58 + "╣")
print("║  PLAYER I:" + " " * 47 + "║")
if pi_indifferent:
    print("║    Indifferent — mix {1, 2} with any p" + " " * 19 + "║")
    print("║    Equilibrium: P(1) = P(2) = 0.50" + " " * 23 + "║")
print("╠" + "═" * 58 + "╣")
print("║  PLAYER II:" + " " * 46 + "║")
print(f"║    H₁ (coin=H, knows i=1): play {h1_result['optimal_j']}" +
      " " * 23 + "║")
print(f"║    H₂ (coin=H, knows i=2): play {h2_result['optimal_j']}" +
      " " * 23 + "║")
t3 = tails_result['pii_mix'][3]
t4 = tails_result['pii_mix'][4]
print(f"║    T  (coin=T, unknown i): P(3)={t3:.2f}, P(4)={t4:.2f}" +
      " " * 11 + "║")
print("╚" + "═" * 58 + "╝")

## 8. Explicit Bellman Equations

Let us write out every Bellman equation in the backward induction.

In [ ]:
print("BELLMAN EQUATIONS FOR THE PARITY GAME")
print("═" * 70)

print("\nSTAGE 4 — TERMINAL PAYOFF (chance node: referee draw)")
print("─" * 70)
for i in PI_ACTIONS:
    for j in PII_ACTIONS:
        terms = [f"{p}·payoff({i},{j},{r})"
                 for r, p in REFEREE_OUTCOMES]
        vals = [f"{p}·({terminal_payoff(i,j,r):+.0f})"
                for r, p in REFEREE_OUTCOMES]
        g = conditional_payoff(i, j)
        print(f"  g({i},{j}) = {' + '.join(terms)}")
        print(f"         = {' + '.join(vals)} = {g:+.1f}")

print("\nSTAGE 3 — PLAYER II DECISION")
print("─" * 70)
for i in PI_ACTIONS:
    res = solve_heads_subgame(i)
    terms = ', '.join([f'g({i},{j})={conditional_payoff(i,j):+.1f}'
                       for j in PII_ACTIONS])
    print(f"  V_H({i}) = min{{ {terms} }} = {res['value']:+.1f}")

print(f"\n  T-subgame (2×2 matrix game):")
print(f"    M = [[g(1,3), g(1,4)], [g(2,3), g(2,4)]]")
print(f"      = [[{M_tails[0,0]:+.1f}, {M_tails[0,1]:+.1f}], "
      f"[{M_tails[1,0]:+.1f}, {M_tails[1,1]:+.1f}]]")
print(f"    Minimax value: V_T = {tails_result['value']:+.2f}")
print(f"    PI* mix at T:  P(1)={tails_result['pi_mix'][1]:.2f}, "
      f"P(2)={tails_result['pi_mix'][2]:.2f}")
print(f"    PII* mix at T: P(3)={tails_result['pii_mix'][3]:.2f}, "
      f"P(4)={tails_result['pii_mix'][4]:.2f}")

print("\nSTAGE 2 — COIN FLIP")
print("─" * 70)
for i in PI_ACTIONS:
    V_H_i = solve_heads_subgame(i)['value']
    print(f"  V_coin({i}) = 0.5·V_H({i}) + 0.5·V_T")
    print(f"           = 0.5·({V_H_i:+.1f}) + 0.5·({V_T:+.1f}) "
          f"= {V_coin[i]:+.2f}")

print("\nSTAGE 1 — PLAYER I DECISION")
print("─" * 70)
terms = ', '.join(f'V_coin({i})={V_coin[i]:+.2f}' for i in PI_ACTIONS)
print(f"  V* = max{{ {terms} }} = {V_star:+.2f}")
if pi_indifferent:
    print(f"  PI indifferent → any p ∈ [0,1] is optimal")

## 9. Full Strategic Form and LP Verification

To cross-check, we also solve the game in **strategic (normal) form**. Player II has $2 \times 2 \times 2 = 8$ pure strategies (one action per info set). We build the $2 \times 8$ payoff matrix and solve the resulting LP.

In [ ]:
def enumerate_pii_strategies():
    """Enumerate PII's 8 pure strategies: (action_H1, action_H2, action_T)."""
    return [(h1, h2, t)
            for h1 in PII_ACTIONS
            for h2 in PII_ACTIONS
            for t in PII_ACTIONS]


def compute_expected_payoff(i: int, pii_strat: tuple) -> float:
    """Expected payoff to PI for PI's action i vs PII's pure strategy."""
    a_h1, a_h2, a_t = pii_strat
    j_heads = a_h1 if i == 1 else a_h2
    j_tails = a_t
    return (COIN_PROB * conditional_payoff(i, j_heads) +
            (1 - COIN_PROB) * conditional_payoff(i, j_tails))


pii_strats = enumerate_pii_strategies()
payoff_matrix = np.array(
    [[compute_expected_payoff(i, s) for s in pii_strats]
     for i in PI_ACTIONS])

print("STRATEGIC FORM: 2 × 8 Payoff Matrix")
print("═" * 70)
header = "PI \\ PII"
print(f"{header:>10}", end="")
for s in pii_strats:
    print(f"{str(s):>9}", end="")
print()
print("─" * 70)
for idx, i in enumerate(PI_ACTIONS):
    print(f"   i = {i}  ", end="")
    for val in payoff_matrix[idx]:
        print(f"{val:>+9.2f}", end="")
    print()

In [ ]:
def solve_matrix_game_lp(M: np.ndarray) -> dict:
    """Solve an m×n zero-sum game via LP (row player's problem).

    max  v
    s.t. Σ_i p_i M[i,j] ≥ v   for all j
         Σ_i p_i = 1,  p_i ≥ 0
    """
    m, n = M.shape

    # Solve by iterating over all vertex solutions of the dual
    # For small games, enumerate all subsets of size m columns
    from itertools import combinations

    best_v = -np.inf
    best_p = None

    for cols in combinations(range(n), m):
        sub = M[:, list(cols)]
        try:
            A = np.vstack([sub.T, np.ones(m)])
            b = np.append(np.zeros(m), 1.0)
            # Solve A p = b in least-squares sense
            p, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
            if np.all(p >= -1e-10):
                p = np.maximum(p, 0)
                p /= p.sum()
                payoffs = p @ M
                v = payoffs.min()
                if v > best_v + 1e-12:
                    best_v = v
                    best_p = p
        except np.linalg.LinAlgError:
            continue

    # Also check pure strategies
    for i in range(m):
        v = M[i, :].min()
        if v > best_v + 1e-12:
            best_v = v
            best_p = np.zeros(m)
            best_p[i] = 1.0

    # PII's best response
    pii_payoffs = best_p @ M
    worst_cols = np.where(np.abs(pii_payoffs - pii_payoffs.min()) < 1e-8)[0]

    return {
        'value': best_v,
        'pi_mix': dict(zip(PI_ACTIONS, best_p.tolist())),
        'pii_active_strats': [pii_strats[c] for c in worst_cols],
    }


lp_result = solve_matrix_game_lp(payoff_matrix)

print("LP SOLUTION (full strategic form)")
print("═" * 60)
print(f"  Game value:  {lp_result['value']:+.4f}")
print(f"  PI mix:      P(1)={lp_result['pi_mix'][1]:.4f}, "
      f"P(2)={lp_result['pi_mix'][2]:.4f}")
print(f"  PII support: {lp_result['pii_active_strats']}")

# Cross-check
assert abs(lp_result['value'] - V_star) < 1e-8, \
    f"LP value {lp_result['value']} != DP value {V_star}"
print(f"\n  ✓ LP value matches backward-induction value: {V_star:+.2f}")

## 10. Visualising the DP Solution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
fig.suptitle('Dynamic Programming: Stage-by-Stage Values',
             fontsize=15, fontweight='bold', color='#89b4fa')

# --- Panel 1: Game tree values ---
ax = axes[0]
stages = ['V*\n(PI choice)', 'V_coin(1)', 'V_coin(2)',
          'V_H(1)\nH₁:play 3', 'V_T\nTails mix',
          'V_H(2)\nH₂:play 4']
values = [V_star, V_coin[1], V_coin[2],
          h1_result['value'], tails_result['value'],
          h2_result['value']]
colors = ['#89b4fa' if v == V_star else
          '#a6e3a1' if v > 0 else
          '#f38ba8' if v < 0 else '#f9e2af' for v in values]

bars = ax.barh(stages[::-1], values[::-1],
               color=colors[::-1], edgecolor='#6c7086',
               height=0.55)
ax.axvline(x=0, color='#6c7086', linewidth=0.8)
ax.set_xlabel('Value (payoff to PI)')
ax.set_title('Game Tree Node Values', color='#89b4fa')
for bar, v in zip(bars, values[::-1]):
    offset = 0.02 if v >= 0 else -0.02
    ax.text(v + offset, bar.get_y() + bar.get_height()/2,
            f'{v:+.2f}', va='center',
            ha='left' if v >= 0 else 'right',
            fontsize=10, fontweight='bold')
ax.set_xlim(-0.8, 0.15)

# --- Panel 2: Payoff matrix heatmap ---
ax = axes[1]
g_arr = np.array([[conditional_payoff(i, j) for j in PII_ACTIONS]
                   for i in PI_ACTIONS])
im = ax.imshow(g_arr, cmap='RdYlGn', vmin=-1, vmax=1, aspect='equal')
ax.set_yticks(range(len(PI_ACTIONS)))
ax.set_yticklabels([f'i = {i}' for i in PI_ACTIONS], fontsize=12)
ax.set_xticks(range(len(PII_ACTIONS)))
ax.set_xticklabels([f'j = {j}' for j in PII_ACTIONS], fontsize=12)
ax.set_title('g(i, j) — Conditional Payoff Matrix', color='#89b4fa')
for ii in range(2):
    for jj in range(2):
        c = 'white' if abs(g_arr[ii,jj]) > 0.5 else 'black'
        ax.text(jj, ii, f'{g_arr[ii,jj]:+.1f}', ha='center',
                va='center', color=c, fontsize=16, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.7)

# --- Panel 3: Optimal strategies ---
ax = axes[2]
info_sets = ['PI\n(action)', 'PII @ H₁', 'PII @ H₂', 'PII @ T']
probs_a = [0.5, 0.0, 0.0,
           tails_result['pii_mix'][3]]
probs_b = [0.5, 1.0, 1.0,
           tails_result['pii_mix'][4]]
labels_a = ['P(1)', 'P(3)', 'P(3)', 'P(3)']
labels_b = ['P(2)', 'P(4)', 'P(4)', 'P(4)']
x = np.arange(len(info_sets))
w = 0.35

bar1 = ax.bar(x - w/2, probs_a, w, color='#a6e3a1',
              edgecolor='#6c7086', label='Action 1/3')
bar2 = ax.bar(x + w/2, probs_b, w, color='#f38ba8',
              edgecolor='#6c7086', label='Action 2/4')

ax.set_xticks(x)
ax.set_xticklabels(info_sets, fontsize=9)
ax.set_ylabel('Probability')
ax.set_title('Optimal Strategies (Equilibrium)', color='#89b4fa')
ax.set_ylim(0, 1.15)
ax.legend(facecolor='#1e1e2e', edgecolor='#6c7086', fontsize=9)

for bars, probs in [(bar1, probs_a), (bar2, probs_b)]:
    for bar, p in zip(bars, probs):
        if p > 0:
            ax.text(bar.get_x() + bar.get_width()/2, p + 0.03,
                    f'{p:.2f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## 11. DP Agents Playing the Game

We create agents that follow the DP-computed optimal strategies and play sample games.

In [ ]:
class ParityGame:
    """Random Disclosure Parity Game environment (for simulation)."""

    def reset(self):
        self.pi_choice = None
        self.coin = None
        self.pii_choice = None
        self.ref_draw = None
        self.done = False
        self.payoff_pi = 0.0
        return self

    def pi_step(self, i: int):
        self.pi_choice = i
        self.coin = 'H' if random.random() < COIN_PROB else 'T'

    def get_pii_info_set(self) -> str:
        if self.coin == 'H':
            return f'H{self.pi_choice}'
        return 'T'

    def pii_step(self, j: int):
        self.pii_choice = j
        r = random.random()
        self.ref_draw = 1 if r < 0.4 else (2 if r < 0.6 else 3)
        total = self.pi_choice + self.pii_choice + self.ref_draw
        self.payoff_pi = 1.0 if total % 2 == 0 else -1.0
        self.done = True


class DPAgentPI:
    """DP-optimal Player I: uniform mix over {1, 2}."""

    def __init__(self, p1: float = 0.5):
        self.p1 = p1

    def select_action(self) -> int:
        return 1 if random.random() < self.p1 else 2


class DPAgentPII:
    """DP-optimal Player II: pure at H1/H2, mixed at T."""

    def __init__(self, h1_action: int, h2_action: int,
                 t_prob_3: float):
        self.h1_action = h1_action
        self.h2_action = h2_action
        self.t_prob_3 = t_prob_3

    def select_action(self, info_set: str) -> int:
        if info_set == 'H1':
            return self.h1_action
        if info_set == 'H2':
            return self.h2_action
        return 3 if random.random() < self.t_prob_3 else 4


dp_pi = DPAgentPI(p1=0.5)
dp_pii = DPAgentPII(
    h1_action=h1_result['optimal_j'],
    h2_action=h2_result['optimal_j'],
    t_prob_3=tails_result['pii_mix'][3])

print("DP agents created:")
print(f"  PI:  mix(1,2) with P(1)={dp_pi.p1:.2f}")
print(f"  PII: H1→{dp_pii.h1_action}, H2→{dp_pii.h2_action}, "
      f"T→mix(3,4) with P(3)={dp_pii.t_prob_3:.2f}")

In [ ]:
def evaluate(pi_fn, pii_fn, num_games: int = 200_000) -> float:
    game = ParityGame()
    total = 0.0
    for _ in range(num_games):
        game.reset()
        i = pi_fn()
        game.pi_step(i)
        info = game.get_pii_info_set()
        j = pii_fn(info)
        game.pii_step(j)
        total += game.payoff_pi
    return total / num_games


random.seed(42)
np.random.seed(42)

print("EVALUATION (200,000 games per matchup)")
print("═" * 60)

# DP vs DP
v1 = evaluate(dp_pi.select_action, dp_pii.select_action)
print(f"  DP PI vs DP PII:                {v1:+.4f}")

# DP PI vs exploitative PII (always plays 3)
v2 = evaluate(dp_pi.select_action, lambda s: 3)
print(f"  DP PI vs PII always-3:          {v2:+.4f}")

# DP PI vs exploitative PII (always plays 4)
v3 = evaluate(dp_pi.select_action, lambda s: 4)
print(f"  DP PI vs PII always-4:          {v3:+.4f}")

# PI always-1 vs DP PII
v4 = evaluate(lambda: 1, dp_pii.select_action)
print(f"  PI always-1 vs DP PII:          {v4:+.4f}")

# PI always-2 vs DP PII
v5 = evaluate(lambda: 2, dp_pii.select_action)
print(f"  PI always-2 vs DP PII:          {v5:+.4f}")

print(f"\n  Theoretical game value:           {V_star:+.2f}")
print(f"\n  Note: DP PI guarantees ≥ {V_star:+.2f} "
      f"regardless of PII's strategy.")
print(f"  DP PII guarantees ≤ {V_star:+.2f} "
      f"regardless of PI's strategy.")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

labels = ['DP PI vs DP PII', 'DP PI vs always-3',
          'DP PI vs always-4', 'always-1 vs DP PII',
          'always-2 vs DP PII']
values_eval = [v1, v2, v3, v4, v5]
bar_colors = ['#89b4fa' if abs(v - V_star) < 0.02 else
              '#a6e3a1' if v > V_star else '#f38ba8'
              for v in values_eval]

bars = ax.barh(labels, values_eval, color=bar_colors,
               edgecolor='#6c7086', height=0.5)
ax.axvline(x=V_star, color='#f9e2af', linestyle='--', linewidth=2,
           label=f'Game value ({V_star:+.2f})')
ax.axvline(x=0, color='#6c7086', linestyle='-', linewidth=0.5)
ax.set_xlabel('Average Payoff to Player I')
ax.set_title('Evaluation: DP Agents vs Various Opponents',
             fontsize=14, fontweight='bold', color='#89b4fa')
ax.legend(loc='lower right', facecolor='#1e1e2e', edgecolor='#6c7086')

for bar, v in zip(bars, values_eval):
    offset = -0.02 if v < 0 else 0.02
    ax.text(v + offset, bar.get_y() + bar.get_height()/2,
            f'{v:+.3f}', ha='right' if v < 0 else 'left',
            va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

## 12. Watching Sample Games

In [ ]:
def play_verbose(pi_fn, pii_fn, pi_name='PI', pii_name='PII',
                 n: int = 12):
    game = ParityGame()
    wins_pi = 0
    for k in range(n):
        game.reset()
        i = pi_fn()
        game.pi_step(i)
        info = game.get_pii_info_set()
        j = pii_fn(info)
        game.pii_step(j)
        s = game.pi_choice + game.pii_choice + game.ref_draw
        par = 'even' if s % 2 == 0 else 'odd'
        winner = pi_name if game.payoff_pi > 0 else pii_name
        if game.payoff_pi > 0:
            wins_pi += 1
        print(f"  Game {k+1:2d}: {pi_name}={i}, coin={game.coin}, "
              f"info={info}, {pii_name}={j}, r={game.ref_draw}, "
              f"sum={s} ({par}) → {winner} wins")
    print(f"\n  {pi_name} won {wins_pi}/{n} "
          f"({100*wins_pi/n:.0f}%)")


random.seed(99)
print("═" * 70)
print("  SAMPLE GAMES: DP PI vs DP PII (both optimal)")
print("═" * 70)
play_verbose(dp_pi.select_action, dp_pii.select_action,
             'DP-PI', 'DP-PII')

random.seed(99)
print(f"\n{'═' * 70}")
print("  SAMPLE GAMES: PI always-1 vs DP PII (exploiting)")
print("═" * 70)
play_verbose(lambda: 1, dp_pii.select_action,
             'Fixed-1', 'DP-PII')

## 13. Sensitivity Analysis: Value of Information

How does the game value change as a function of $P(\text{heads})$?

In [ ]:
def game_value_for_coin_prob(p_heads: float) -> float:
    """Compute the exact game value for a given P(heads)."""
    # V_H(1) and V_H(2) are fixed (independent of coin prob)
    V_H_1 = min(conditional_payoff(1, j) for j in PII_ACTIONS)
    V_H_2 = min(conditional_payoff(2, j) for j in PII_ACTIONS)

    # V_T (tails subgame value) is also fixed
    M = np.array([[conditional_payoff(i, j) for j in PII_ACTIONS]
                   for i in PI_ACTIONS])
    t_res = solve_2x2_matrix_game(M)
    V_T_val = t_res['value']

    # V_coin(i) = p_heads * V_H(i) + (1 - p_heads) * V_T
    V_c1 = p_heads * V_H_1 + (1 - p_heads) * V_T_val
    V_c2 = p_heads * V_H_2 + (1 - p_heads) * V_T_val
    return max(V_c1, V_c2)


p_range = np.linspace(0, 1, 200)
v_range = [game_value_for_coin_prob(p) for p in p_range]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(p_range, v_range, color='#89b4fa', linewidth=2.5)
ax.axhline(y=0, color='#6c7086', linewidth=0.8)
ax.axvline(x=0.5, color='#f9e2af', linestyle='--', linewidth=1.5,
           label='P(H) = 0.5 (our game)')
ax.plot(0.5, V_star, 'o', color='#f38ba8', markersize=10, zorder=5,
        label=f'V* = {V_star:+.2f}')

ax.fill_between(p_range, v_range, 0,
                where=[v < 0 for v in v_range],
                alpha=0.15, color='#f38ba8')
ax.fill_between(p_range, v_range, 0,
                where=[v >= 0 for v in v_range],
                alpha=0.15, color='#a6e3a1')

ax.set_xlabel('P(Heads) — probability of disclosure', fontsize=12)
ax.set_ylabel('Game value V*', fontsize=12)
ax.set_title('Value of Information: Game Value vs Disclosure Probability',
             fontsize=14, fontweight='bold', color='#89b4fa')
ax.legend(loc='lower left', facecolor='#1e1e2e', edgecolor='#6c7086')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 1)

ax.annotate('More disclosure\nhurts PI', xy=(0.8, -0.45),
            fontsize=10, color='#f38ba8', ha='center')
ax.annotate('No disclosure:\nfair matching\npennies',
            xy=(0.03, 0.02), fontsize=9, color='#a6e3a1')

plt.tight_layout()
plt.show()

print("\nKey observations:")
print(f"  P(H)=0.0 → V* = {game_value_for_coin_prob(0):+.2f} "
      f"(no disclosure: fair game)")
print(f"  P(H)=0.5 → V* = {game_value_for_coin_prob(0.5):+.2f} "
      f"(our game)")
print(f"  P(H)=1.0 → V* = {game_value_for_coin_prob(1):+.2f} "
      f"(full disclosure: PII always matches)")
print(f"\n  Information disclosure hurts Player I because PII\n"
      f"  can exploit knowledge of PI's choice to match parity.")

## 14. Comparison: DP vs Learning Methods

Dynamic Programming provides the **exact** solution, while learning methods (Q-learning, Monte Carlo, gradient bandits) only approximate it through sampling.

In [ ]:
comparison = """
╔═════════════════════════════════════════════════════════════════════════════════╗
║              COMPARISON: DP vs LEARNING METHODS FOR THE PARITY GAME             ║
╠═════════════════════════════════════════════════════════════════════════════════╣
║ Aspect              │ Dynamic Programming    │ Q-Learning / MC / Grad Bandit    ║
╠═════════════════════════════════════════════════════════════════════════════════╣
║ Model Required      │ Yes (full game tree)   │ No (learns from samples)         ║
║ Solution Quality    │ Exact optimal          │ Approximate (converges)          ║
║ Handles Imperfect   │ Via subgame LP/        │ Naturally via info sets           ║
║   Information       │   matrix game solver   │   and empirical frequencies      ║
║ Mixed Strategies    │ Computed analytically  │ Emerge from exploration/          ║
║                     │   via minimax theorem  │   empirical action averages      ║
║ Computation         │ O(|strategies|²)       │ O(episodes × steps)              ║
║ Online Learning     │ No                     │ Yes                              ║
║ Scales to Large     │ Exponential in         │ Can use function                 ║
║   Games             │   |info sets|          │   approximation                  ║
╚═════════════════════════════════════════════════════════════════════════════════╝
"""
print(comparison)

print("For the Random Disclosure Parity Game:")
print("─" * 60)
print(f"  PI strategies:     {len(PI_ACTIONS)}")
print(f"  PII strategies:    {len(pii_strats)} "
      f"(2 actions × 3 info sets)")
print(f"  Payoff matrix:     {len(PI_ACTIONS)} × {len(pii_strats)}")
print(f"  DP solved in:      4 stages of backward induction")
print(f"  Learning methods:  ~300,000–500,000 episodes")
print(f"\n  DP is vastly more efficient for this small game,")
print(f"  but requires knowing the game tree structure and")
print(f"  all probabilities (coin, referee) in advance.")

## 15. Conclusion

### Summary

We solved the Random Disclosure Parity Game **exactly** using dynamic programming — specifically, **backward induction** through four stages of the game tree.

**Key results:**

1. **Stage 4 (referee draw)**: Computed the conditional expected payoff $g(i,j) = \pm 0.6$ — same-parity pairs $(i,j)$ yield $-0.6$ (PII wins), opposite-parity yields $+0.6$ (PI wins).

2. **Stage 3 (PII's decision)**:
   - $H_1$: PII plays 3 (match parity), value $= -0.6$
   - $H_2$: PII plays 4 (match parity), value $= -0.6$
   - $T$: 2×2 matching-pennies subgame → both mix 50/50, value $= 0$

3. **Stage 2 (coin flip)**: $V_\text{coin}(i) = 0.5 \cdot (-0.6) + 0.5 \cdot 0 = -0.30$ for both $i$.

4. **Stage 1 (PI's choice)**: PI is indifferent → $V^* = -0.30$.

### DP for Imperfect-Information Games

Unlike the Addition game where standard backward induction suffices, this game required:

- **Chance-node integration**: Expected values over the referee draw and coin flip.
- **Matrix game solver**: At information set $T$, PII's lack of knowledge creates a simultaneous-move subgame that must be solved via the minimax theorem (analytically for 2×2, LP for larger games).
- **Indifference verification**: PI's indifference between actions 1 and 2 implies any mixture is optimal — the equilibrium is not unique in PI's strategy.

### Value of Information

The sensitivity analysis reveals that disclosure **always hurts Player I** in this game: more information to PII allows better parity matching, driving the game value down from $0$ (no disclosure) to $-0.6$ (full disclosure).

### Connections to Blackwell's Work

Dynamic programming for zero-sum games is the computational realisation of the **minimax theorem** (von Neumann, 1928). Blackwell's contributions extended this to sequential decision problems — the Bellman equations we solved are the finite-game analogue of Blackwell's optimality equations for Markov decision processes.